# Assignment 4 - Part II: Diagnosis

Diagnosis is the second step of the PHM loop: after detecting that something is off, identify what kind of state the asset is in. Here, that means unsupervised clustering of the MNIST images from Part I to see whether K-Means can separate the digit classes on its own.

Following the assignment specifications literally: *use K-Means with numpy and matplotlib*, clustering runs on the raw flattened pixel vectors (784-dim), not on autoencoder latents.

## What this notebook covers for the different grades

- **Grade 3:** discern different classes in the Part I dataset. Spec says *the 8 different figures*; treated as a typo since normal MNIST is digits 0–8 (9 classes). Using the full train set (digits 0–9) here so the comparison stays honest.
- **Grade 4:** fit K-Means on the flattened MNIST pixels at a fixed `k`, then map clusters to digit labels by majority vote.
- **Grade 5:** sweep `k`, plot inertia vs k (the elbow), and pick the optimal `k` geometrically. Build the 10×10 confusion matrix at the chosen `k`.

In [ ]:
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
import torch
from torchvision import datasets, transforms
from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix
from tqdm.auto import tqdm

from config import *

In [ ]:
print(f"Python    : {sys.version.split()[0]}")
print(f"PyTorch   : {torch.__version__}")
print(f"CUDA      : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device    : {torch.cuda.get_device_name(0)}")
print(f"Conda env : {os.environ.get('CONDA_DEFAULT_ENV', 'unknown')}")

In [ ]:
MNIST_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)

print(f"k range for elbow : {KMEANS_K_RANGE[0]}..{KMEANS_K_RANGE[-1]}")
print(f"elbow sample size : {KMEANS_SAMPLE_SIZE}")
print(f"n_init            : {KMEANS_N_INIT}")

## Grade 3 - load MNIST and flatten

Same dataset as Part I, same normalisation. K-Means works on flat numpy arrays, so each `(1, 28, 28)` image becomes a 784-dim vector.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MNIST_MEAN, MNIST_STD),
])

mnist_train = datasets.MNIST(root=str(MNIST_DIR), train=True, download=True, transform=transform)

# Pull every image into one big numpy array. Flat 784-dim vectors per image.
imgs   = mnist_train.data.float() / 255.0          # (60000, 28, 28), [0, 1]
imgs   = (imgs - MNIST_MEAN[0]) / MNIST_STD[0]     # match transform
X_all  = imgs.view(imgs.size(0), -1).numpy()       # (60000, 784)
y_all  = mnist_train.targets.numpy()               # (60000,)

print(f"X_all shape: {X_all.shape}, y_all shape: {y_all.shape}")
print(f"class counts: {np.bincount(y_all)}")

In [ ]:
# Subsample for the elbow sweep.
# Raw 784-dim K-Means over 60k points x many values of k is painful, and the elbow shape is robust to subsampling.
rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(X_all), size=min(KMEANS_SAMPLE_SIZE, len(X_all)), replace=False)
X_sample = X_all[sample_idx]
print(f"sample for elbow: {X_sample.shape}")

## Grade 5 - elbow analysis (run first so the chosen `k` drives the rest)

Inertia is the within-cluster sum of squared distances. It always drops as `k` grows, so we look for the elbow - the point past which the drop becomes marginal.

In [ ]:
k_values, inertia_values = [], []
for k in tqdm(KMEANS_K_RANGE, desc="elbow sweep", unit="k"):
    km = KMeans(n_clusters=k, random_state=SEED, n_init=KMEANS_N_INIT)
    km.fit(X_sample)
    k_values.append(int(k))
    inertia_values.append(float(km.inertia_))

In [ ]:
# Elbow = farthest point from the chord between the first and last (k, inertia).
# Pure geometry, no extra dependency.
pts = np.column_stack([np.asarray(k_values, dtype=float),
                       np.asarray(inertia_values, dtype=float)])
start, end = pts[0], pts[-1]
line = end - start
line_norm = line / np.linalg.norm(line)
vecs = pts - start
proj = vecs @ line_norm
dists = np.linalg.norm(vecs - np.outer(proj, line_norm), axis=1)
optimal_k = int(k_values[int(np.argmax(dists))])
print(f"optimal k (elbow): {optimal_k}")

## Grade 4 - fit K-Means at the chosen `k` on the full dataset

In [ ]:
kmeans = KMeans(n_clusters=optimal_k, random_state=SEED, n_init=KMEANS_N_INIT)
kmeans.fit(X_all)
cluster_ids = kmeans.predict(X_all)
print(f"fitted K-Means with k={optimal_k} on {len(X_all)} samples")

In [ ]:
# Map each cluster to a digit by majority vote, so the confusion matrix is
# interpretable in digit space rather than arbitrary cluster IDs.
cluster_to_digit = {}
for c in range(optimal_k):
    members = y_all[cluster_ids == c]
    if len(members) == 0:
        continue
    cluster_to_digit[c] = int(np.bincount(members).argmax())

y_pred = np.array([cluster_to_digit[c] for c in cluster_ids])
accuracy = float((y_pred == y_all).mean())
print(f"majority-vote accuracy at k={optimal_k}: {accuracy:.4f}")
print(f"cluster -> digit map: {cluster_to_digit}")

## What I can extrapolate from this

Raw-pixel K-Means is a hard baseline for MNIST: digits with similar pen strokes (4/9, 3/8, 7/1) often share clusters because Euclidean distance in pixel space measures shape overlap more than digit identity. The elbow tends to land near the true class count, but the majority-vote accuracy stays modest because some digits split across multiple clusters while others merge.

In [ ]:
# Grade 5 requires an elbow plot with optimal_k marker and the 10x10 confusion matrix heatmap
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
ax.plot(k_values, inertia_values, marker="o", linewidth=1.5)
ax.axvline(optimal_k, color="red", linestyle="--", label=f"elbow @ k={optimal_k}")
ax.set_xlabel("k (number of clusters)")
ax.set_ylabel("Inertia (within-cluster SSE)")
ax.set_title("K-Means inertia vs k")
ax.grid(alpha=0.3); 
ax.legend()

cm = confusion_matrix(y_all, y_pred, labels=list(range(10)))

ax = axes[1]
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(10)); 
ax.set_yticks(range(10))
ax.set_xlabel("predicted digit (via cluster->majority map)")
ax.set_ylabel("true digit")
ax.set_title(f"Confusion matrix (k={optimal_k}, acc={accuracy:.3f})")

for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7,
                color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout(); plt.show()